In [1]:
import math
import numpy as np
from collections import Counter

In [2]:
# Get Data

In [3]:
docs = {
    "d1" : "scary green crocodile",
    "d2" : "scary green big",
    "d3" : "small crocodile",
}
print(docs)

{'d1': 'scary green crocodile', 'd2': 'scary green big', 'd3': 'small crocodile'}


In [4]:
# 2. Preprocessing

In [5]:
def tokenize(text):
    return text.split()

tokenized_docs = {}
for d, text in docs.items():
    tokens = tokenize(text)
    tokenized_docs[d] = tokens
print(tokenized_docs)

{'d1': ['scary', 'green', 'crocodile'], 'd2': ['scary', 'green', 'big'], 'd3': ['small', 'crocodile']}


In [6]:
all_words = []
for doc in tokenized_docs.values():
    for word in doc:
        all_words.append(word)
print(all_words)

unique_words = set(all_words)
print(unique_words)
vocab = sorted(unique_words)
print(vocab)

['scary', 'green', 'crocodile', 'scary', 'green', 'big', 'small', 'crocodile']
{'small', 'big', 'scary', 'crocodile', 'green'}
['big', 'crocodile', 'green', 'scary', 'small']


In [7]:
tf = {}
for d, words in tokenized_docs.items():
    counts = Counter(words)
    print(d, counts)

    tf_row = []
    for terms in vocab:
        if terms in counts:
            tf_row.append(counts[terms])
        else:
            tf_row.append(0)
    tf[d] = tf_row
    print(tf)

d1 Counter({'scary': 1, 'green': 1, 'crocodile': 1})
{'d1': [0, 1, 1, 1, 0]}
d2 Counter({'scary': 1, 'green': 1, 'big': 1})
{'d1': [0, 1, 1, 1, 0], 'd2': [1, 0, 1, 1, 0]}
d3 Counter({'small': 1, 'crocodile': 1})
{'d1': [0, 1, 1, 1, 0], 'd2': [1, 0, 1, 1, 0], 'd3': [0, 1, 0, 0, 1]}


In [8]:
print("\nTF Matrix (rows=docs, columns=terms):")
for d in docs:
    print(d, tf[d])


TF Matrix (rows=docs, columns=terms):
d1 [0, 1, 1, 1, 0]
d2 [1, 0, 1, 1, 0]
d3 [0, 1, 0, 0, 1]


In [9]:
# TF & IDF

In [10]:
N= len(docs)
print("Number of Documents:", N)


Number of Documents: 3


In [11]:
df = {}
for terms in vocab:
    counter = 0
    for d in docs:
        words_in_doc = tokenized_docs[d]
        if terms in words_in_doc:
            counter += 1
        df[terms] = counter

print(df)

{'big': 1, 'crocodile': 2, 'green': 2, 'scary': 2, 'small': 1}


In [12]:
N= len(docs)
idf = {}
for terms in vocab:
    df_value = df[terms]
    idf_value = math.log2(N/df_value)
    idf[terms] = idf_value
print("\nIDF Matrix (rows=docs, columns=terms):")
for terms in idf:
    print(terms, idf[terms])


IDF Matrix (rows=docs, columns=terms):
big 1.584962500721156
crocodile 0.5849625007211562
green 0.5849625007211562
scary 0.5849625007211562
small 1.584962500721156


In [13]:
# TF-IDF Matrix

In [14]:
tfidf = {}
for doc in docs:
    tfidf_row= []
    for i in range(len(vocab)):
        term = vocab[i]
        tf_value = tf[doc][i]
        idf_value = idf[term]
        thidf_weight = tf_value * idf_value
        tfidf_row.append(thidf_weight)
    tfidf[doc] = tfidf_row

In [15]:
print("\nTf-IDF Matrix (rows=docs, columns=Vocabulary):")
for terms in tfidf:
    print(terms, tfidf[terms])


Tf-IDF Matrix (rows=docs, columns=Vocabulary):
d1 [0.0, 0.5849625007211562, 0.5849625007211562, 0.5849625007211562, 0.0]
d2 [1.584962500721156, 0.0, 0.5849625007211562, 0.5849625007211562, 0.0]
d3 [0.0, 0.5849625007211562, 0.0, 0.0, 1.584962500721156]


In [16]:
# ESA Vector for Input Text

In [17]:
def esa_vec(text):
    words = tokenize(text)
    q_tf  = Counter(words)
    print(q_tf)
    q_vec = []
    for term in vocab:
        if term in q_tf:
            q_vec.append(q_tf[term] * idf[term])
        else:
            q_vec.append(0)

    print(q_vec)

    # Concept score one for per document
    concept_score = []
    for doc in docs:
        score = sum(q_vec[i] * tfidf[doc][i] for i in range(len(vocab)))
        concept_score.append(score)

    return concept_score


In [18]:
v= esa_vec("green crocodile")
print("\n ESA Vector for 'green crocodile' :",np.round(v,3))

Counter({'green': 1, 'crocodile': 1})
[0, 0.5849625007211562, 0.5849625007211562, 0, 0]

 ESA Vector for 'green crocodile' : [0.684 0.342 0.342]


In [19]:
# Cosine Similarity

In [20]:
def cosine_relatedness(v1,v2):
    return np.dot(v1,v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

v1_vec_score = esa_vec("big crocodile")
v2_vec_score = esa_vec("scary crocodile")

sim = cosine_relatedness(v1_vec_score, v2_vec_score)
print("\n ESA Vector for 'big crocodile' :",np.round(v1_vec_score,3))
print("\n ESA Vector for 'scary crocodile' :",np.round(v2_vec_score,3))
print("Cosine Similarity: ",np.round(sim,3))

Counter({'big': 1, 'crocodile': 1})
[1.584962500721156, 0.5849625007211562, 0, 0, 0]
Counter({'scary': 1, 'crocodile': 1})
[0, 0.5849625007211562, 0, 0.5849625007211562, 0]

 ESA Vector for 'big crocodile' : [0.342 2.512 0.342]

 ESA Vector for 'scary crocodile' : [0.684 0.342 0.342]
Cosine Similarity:  0.565
